In [1]:
import os
import pandas as pd
import datajoint as dj
# from datetime import datenid
from datetime import datetime
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sb

c:\Users\bramn\Anaconda3\envs\analysis\lib\site-packages\numpy\_distributor_init.py:32: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\bramn\Anaconda3\envs\analysis\lib\site-packages\numpy\.libs\libopenblas.PYQHXLVVQ7VESDPUVUADXEVJOBGHJPAY.gfortran-win_amd64.dll
c:\Users\bramn\Anaconda3\envs\analysis\lib\site-packages\numpy\.libs\libopenblas.XWYDX2IKJW2NMTWSFYNGFUWKQU3LYTCZ.gfortran-win_amd64.dll
  stacklevel=1)


In [2]:
dj.config['database.host'] = '10.38.76.219'
dj.config['database.user'] = 'root'
dj.config['database.password'] = 'retina'
dj.config["enable_python_native_blobs"] = True # Only for testing in dj0.12 (https://github.com/datajoint/datajoint-python#python-native-blobs)

%env DJ_SUPPORT_FILEPATH_MANAGEMENT=TRUE
import os
os.chdir('../..')
import main

drive_path = r"V:\Data\datajoint\DJEphys"
data_path = r"T:\Data"

ft = main.load(drive_path, data_path)

env: DJ_SUPPORT_FILEPATH_MANAGEMENT=TRUE
Connecting root@10.38.76.219:3306


In [3]:
exp_table = ft.Experiment()
sess_table = ft.Session()
mouse_table = ft.Mouse()
subsess_table = ft.Subsession()
raw_table = ft.ephys_tables.EphysRaw()
raw_helper_table = ft.ephys_tables.EphysRawHelper()
events_table = ft.event_tables.SubsessionEvents()
sess_split_table = ft.SessionSplit() 
sorted_table = ft.ephys_tables.SpikeSorted()
metrics_table = ft.ephys_tables.ClusterMetrics()
subsess_spikes_table = ft.ephys_tables.SubsessionSpikes()
trials_table = ft.ephys_tables.TrialSpikes()
opto_table = ft.ephys_tables.OptoTagging()
opto_cluster_table = ft.ephys_tables.OptoCluster()
cluster_table = ft.ephys_tables.Cluster()
opto_clustering_table = ft.ephys_tables.OptoClusterClustering()
cluster_trials_table = ft.ephys_tables.ClusterTrial()
behavior_table = ft.behavior_tables.Behavioral()
behavior_trials_table = ft.behavior_tables.TrialBehavioral()
cluster_sweep_metrics_table = ft.ephys_analysis_tables.ClusterSweepMetrics()
cluster_loom_metrics_table = ft.ephys_analysis_tables.ClusterLoomMetrics()
cluster_depth_table = ft.ephys_tables.ClusterDepth()
cluster_trial_sweeping_table = ft.ephys_tables.TrialSweeping()
cluster_trials2_table = ft.ephys_tables.ClusterTrial2()
opto_curation_table = ft.ephys_tables.OptoCuration()


In [4]:
from datajoint_utils import get_cluster_frs
# stim_types = ['EXPA_25', 'EXPA_60', 'EXPA', 'EXPW', 'EXPW_60', 'DIMM', 'SWEP_4', 'SWEP_12', 'SWEP_20', 'SWEP_50']
stim_types = ['EXPA', 'EXPA_60', 'EXPA_25', 'EXPW', 'EXPW_60','DIMM', 'DIMM_contr', 'SWEP_4', 'SWEP_12', 'SWEP_20', 'SWEP_50']
# stim_types = ['EXPA', 'EXPA_60', 'EXPA_25', 'EXPW', 'EXPW_60','DIMM', 'SWEP_4', 'SWEP_12', 'SWEP_20', 'SWEP_50', 'SWEP_4_extra', 'SWEP_12_extra', 'SWEP_20_extra', 'SWEP_50_extra'] # 'SWEP_4', 'SWEP_12', 'SWEP_20',


In [5]:
import datasets

ds = datasets.get_dataset('population_coding', as_keys=True)
SC_recs_dict = ds['SC']
LP_recs_dict = ds['LP']

In [6]:
# -------------- STATIC ONLY - VR - ALL --------------
lm_clusters = cluster_table.aggr((cluster_loom_metrics_table), lm_zeta_p="min(lm_zeta_p)")
sm_clusters = cluster_table.aggr(cluster_sweep_metrics_table, sm_zeta_p="min(sm_zeta_p)")
all_clusters = lm_clusters * sm_clusters & 'lm_zeta_p<0.05 OR sm_zeta_p<0.05'
# all_clusters = lm_clusters & 'lm_zeta_p<0.05'

sSC_clusters = (cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'sSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_clusters = (cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'dSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
LP_clusters = (cluster_table * cluster_depth_table & LP_recs_dict & {'area': 'LP'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')

sSC_keys = (all_clusters & sSC_clusters).fetch('KEY')
dSC_keys = (all_clusters & dSC_clusters).fetch('KEY')
LP_keys = (all_clusters & LP_clusters).fetch('KEY')

print("sSC:", len(sSC_keys), ", dSC:", len(dSC_keys), ", LP:", len(LP_keys))

sSC_dict = get_cluster_frs(sSC_keys, stim_types, dataset_name='sSC_VR_loomsweep_all_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)
dSC_dict = get_cluster_frs(dSC_keys, stim_types, dataset_name='dSC_VR_loomsweep_all_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)
LP_dict = get_cluster_frs(LP_keys, stim_types, dataset_name='LP_VR_loomsweep_all_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)


Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 366}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 366}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 366}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 

In [ ]:
# -------------- STATIC ONLY - VR - GOOD --------------
lm_clusters = cluster_table.aggr((cluster_loom_metrics_table), lm_zeta_p="min(lm_zeta_p)")
# sm_clusters = cluster_table.aggr(cluster_sweep_metrics_table, sm_zeta_p="min(sm_zeta_p)")
# all_clusters = lm_clusters * sm_clusters & 'lm_zeta_p<0.05 OR sm_zeta_p<0.05'
all_clusters = lm_clusters & 'lm_zeta_p<0.05'

sSC_clusters = (cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'sSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_clusters = (cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'dSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
LP_clusters = (cluster_table * cluster_depth_table & LP_recs_dict & {'area': 'LP', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')

sSC_keys = (all_clusters & sSC_clusters).fetch('KEY')
dSC_keys = (all_clusters & dSC_clusters).fetch('KEY')
LP_keys = (all_clusters & LP_clusters).fetch('KEY')

print("sSC:", len(sSC_keys), ", dSC:", len(dSC_keys), ", LP:", len(LP_keys))

sSC_dict = get_cluster_frs(sSC_keys, stim_types, dataset_name='sSC_VR_loomsweep_good_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)
dSC_dict = get_cluster_frs(dSC_keys, stim_types, dataset_name='dSC_VR_loomsweep_good_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)
LP_dict = get_cluster_frs(LP_keys, stim_types, dataset_name='LP_VR_loomsweep_good_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)



sSC: 292 , dSC: 545 , LP: 564
Data not found locally, extracting from datajoint...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 556}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 556}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 556}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,c

In [8]:
# -------------- STATIC ONLY - ALL --------------
lm_clusters = cluster_table.aggr((cluster_loom_metrics_table), lm_zeta_p="min(lm_zeta_p)")
sm_clusters = cluster_table.aggr(cluster_sweep_metrics_table, sm_zeta_p="min(sm_zeta_p)")
all_clusters = lm_clusters * sm_clusters
# all_clusters = lm_clusters & 'lm_zeta_p<0.05'

sSC_clusters = (cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'sSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_clusters = (cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'dSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
LP_clusters = (cluster_table * cluster_depth_table & LP_recs_dict & {'area': 'LP'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')

sSC_keys = (all_clusters & sSC_clusters).fetch('KEY')
dSC_keys = (all_clusters & dSC_clusters).fetch('KEY')
LP_keys = (all_clusters & LP_clusters).fetch('KEY')

print("sSC:", len(sSC_keys), ", dSC:", len(dSC_keys), ", LP:", len(LP_keys))

sSC_dict = get_cluster_frs(sSC_keys, stim_types, dataset_name='sSC_all_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)
dSC_dict = get_cluster_frs(dSC_keys, stim_types, dataset_name='dSC_all_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)
LP_dict = get_cluster_frs(LP_keys, stim_types, dataset_name='LP_all_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)


sSC: 1328 , dSC: 1710 , LP: 1487
Data not found locally, extracting from datajoint...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 539}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 539}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 539}
Found data for subsession 02, combining...
Stimulus protocol was potentially split u

In [9]:
# -------------- STATIC ONLY - ALL --------------
lm_clusters = cluster_table.aggr((cluster_loom_metrics_table), lm_zeta_p="min(lm_zeta_p)")
sm_clusters = cluster_table.aggr(cluster_sweep_metrics_table, sm_zeta_p="min(sm_zeta_p)")
all_clusters = lm_clusters * sm_clusters
# all_clusters = lm_clusters & 'lm_zeta_p<0.05'

sSC_clusters = (cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'sSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_clusters = (cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'dSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
LP_clusters = (cluster_table * cluster_depth_table & LP_recs_dict & {'area': 'LP', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')

sSC_keys = (all_clusters & sSC_clusters).fetch('KEY')
dSC_keys = (all_clusters & dSC_clusters).fetch('KEY')
LP_keys = (all_clusters & LP_clusters).fetch('KEY')

print("sSC:", len(sSC_keys), ", dSC:", len(dSC_keys), ", LP:", len(LP_keys))

sSC_dict = get_cluster_frs(sSC_keys, stim_types, dataset_name='sSC_good_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)
dSC_dict = get_cluster_frs(dSC_keys, stim_types, dataset_name='dSC_good_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)
LP_dict = get_cluster_frs(LP_keys, stim_types, dataset_name='LP_good_fr25_20260116', means=False, subsession_id='01', fix_sweeping=False)


sSC: 329 , dSC: 890 , LP: 975
Data not found locally, extracting from datajoint...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 553}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 553}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,checking whether can be combining... {'experiment_id': '02218', 'mouse_id': '113L', 'session_id': '20240409', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 553}
Found data for subsession 02, combining...
Stimulus protocol was potentially split up,c

In [ ]:
# -------------- STATIC ONLY - VR - GOOD - INHIBITION --------------
stim_types = ['EXPA', 'EXPA_60', 'EXPA_25', 'EXPW', 'EXPW_60','DIMM']

ds = datasets.get_dataset('inhibition', as_keys=True)
SC_GRP_CNO_recs_dict = ds[f'GRP_SC_CNO']
SC_GRP_CTRL_recs_dict = ds['GRP_SC_CTRL']
SC_NTSR_CNO_recs_dict = ds[f'NTSR_SC_CNO']
SC_NTSR_CTRL_recs_dict = ds['NTSR_SC_CTRL']
LP_NTSR_CNO_recs_dict = ds[f'NTSR_LP_CNO']
LP_NTSR_CTRL_recs_dict = ds['NTSR_LP_CTRL']

lm_clusters = cluster_table.aggr((cluster_loom_metrics_table), lm_zeta_p="min(lm_zeta_p)")
all_clusters = lm_clusters & 'lm_zeta_p<0.05'

sSC_GRP_CNO_clusters = (cluster_table * cluster_depth_table & SC_GRP_CNO_recs_dict & {'area': 'sSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
sSC_GRP_CTRL_clusters = (cluster_table * cluster_depth_table & SC_GRP_CTRL_recs_dict & {'area': 'sSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_GRP_CNO_clusters = (cluster_table * cluster_depth_table & SC_GRP_CNO_recs_dict & {'area': 'dSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_GRP_CTRL_clusters = (cluster_table * cluster_depth_table & SC_GRP_CTRL_recs_dict & {'area': 'dSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
sSC_NTSR_CNO_clusters = (cluster_table * cluster_depth_table & SC_NTSR_CNO_recs_dict & {'area': 'sSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
sSC_NTSR_CTRL_clusters = (cluster_table * cluster_depth_table & SC_NTSR_CTRL_recs_dict & {'area': 'sSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_NTSR_CNO_clusters = (cluster_table * cluster_depth_table & SC_NTSR_CNO_recs_dict & {'area': 'dSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_NTSR_CTRL_clusters = (cluster_table * cluster_depth_table & SC_NTSR_CTRL_recs_dict & {'area': 'dSC', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
LP_NTSR_CNO_clusters = (cluster_table * cluster_depth_table & LP_NTSR_CNO_recs_dict & {'area': 'LP', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
LP_NTSR_CTRL_clusters = (cluster_table * cluster_depth_table & LP_NTSR_CTRL_recs_dict & {'area': 'LP', 'auto_group': 'good'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')

sSC_GRP_CNO_keys = (all_clusters & sSC_GRP_CNO_clusters).fetch('KEY')
sSC_GRP_CTRL_keys = (all_clusters & sSC_GRP_CTRL_clusters).fetch('KEY')
dSC_GRP_CNO_keys = (all_clusters & dSC_GRP_CNO_clusters).fetch('KEY')
dSC_GRP_CTRL_keys = (all_clusters & dSC_GRP_CTRL_clusters).fetch('KEY')

sSC_NTSR_CNO_keys = (all_clusters & sSC_NTSR_CNO_clusters).fetch('KEY')
sSC_NTSR_CTRL_keys = (all_clusters & sSC_NTSR_CTRL_clusters).fetch('KEY')
dSC_NTSR_CNO_keys = (all_clusters & dSC_NTSR_CNO_clusters).fetch('KEY')
dSC_NTSR_CTRL_keys = (all_clusters & dSC_NTSR_CTRL_clusters).fetch('KEY')
LP_NTSR_CNO_keys = (all_clusters & LP_NTSR_CNO_clusters).fetch('KEY')
LP_NTSR_CTRL_keys = (all_clusters & LP_NTSR_CTRL_clusters).fetch('KEY')

print("sSC_GRP_CNO_keys:", len(sSC_GRP_CNO_keys), ", sSC_GRP_CTRL_keys:", len(sSC_GRP_CTRL_keys))
print("dSC_GRP_CNO_keys:", len(dSC_GRP_CNO_keys), ", dSC_GRP_CTRL_keys:", len(dSC_GRP_CTRL_keys))
print("sSC_NTSR_CNO_keys:", len(sSC_NTSR_CNO_keys), ", sSC_NTSR_CTRL_keys:", len(sSC_NTSR_CTRL_keys))
print("dSC_NTSR_CNO_keys:", len(dSC_NTSR_CNO_keys), ", dSC_NTSR_CTRL_keys:", len(dSC_NTSR_CTRL_keys))
print("LP_NTSR_CNO_keys:", len(LP_NTSR_CNO_keys), ", LP_NTSR_CTRL_keys:", len(LP_NTSR_CTRL_keys))

sSC_GRP_CNO_pre_dict = get_cluster_frs(sSC_GRP_CNO_keys, stim_types, dataset_name='sSC_GRP_CNO_pre_VR_good_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
sSC_GRP_CNO_post_dict = get_cluster_frs(sSC_GRP_CNO_keys, stim_types, dataset_name='sSC_GRP_CNO_post_VR_good_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
sSC_GRP_CTRL_pre_dict = get_cluster_frs(sSC_GRP_CTRL_keys, stim_types, dataset_name='sSC_GRP_CTRL_pre_VR_good_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
sSC_GRP_CTRL_post_dict = get_cluster_frs(sSC_GRP_CTRL_keys, stim_types, dataset_name='sSC_GRP_CTRL_post_VR_good_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
dSC_GRP_CNO_pre_dict = get_cluster_frs(dSC_GRP_CNO_keys, stim_types, dataset_name='dSC_GRP_CNO_pre_VR_good_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
dSC_GRP_CNO_post_dict = get_cluster_frs(dSC_GRP_CNO_keys, stim_types, dataset_name='dSC_GRP_CNO_post_VR_good_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
dSC_GRP_CTRL_pre_dict = get_cluster_frs(dSC_GRP_CTRL_keys, stim_types, dataset_name='dSC_GRP_CTRL_pre_VR_good_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
dSC_GRP_CTRL_post_dict = get_cluster_frs(dSC_GRP_CTRL_keys, stim_types, dataset_name='dSC_GRP_CTRL_post_VR_good_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)

sSC_NTSR_CNO_pre_dict = get_cluster_frs(sSC_NTSR_CNO_keys, stim_types, dataset_name='sSC_NTSR_CNO_pre_VR_good_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
sSC_NTSR_CNO_post_dict = get_cluster_frs(sSC_NTSR_CNO_keys, stim_types, dataset_name='sSC_NTSR_CNO_post_VR_good_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
sSC_NTSR_CTRL_pre_dict = get_cluster_frs(sSC_NTSR_CTRL_keys, stim_types, dataset_name='sSC_NTSR_CTRL_pre_VR_good_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
sSC_NTSR_CTRL_post_dict = get_cluster_frs(sSC_NTSR_CTRL_keys, stim_types, dataset_name='sSC_NTSR_CTRL_post_VR_good_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
dSC_NTSR_CNO_pre_dict = get_cluster_frs(dSC_NTSR_CNO_keys, stim_types, dataset_name='dSC_NTSR_CNO_pre_VR_good_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
dSC_NTSR_CNO_post_dict = get_cluster_frs(dSC_NTSR_CNO_keys, stim_types, dataset_name='dSC_NTSR_CNO_post_VR_good_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
dSC_NTSR_CTRL_pre_dict = get_cluster_frs(dSC_NTSR_CTRL_keys, stim_types, dataset_name='dSC_NTSR_CTRL_pre_VR_good_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
dSC_NTSR_CTRL_post_dict = get_cluster_frs(dSC_NTSR_CTRL_keys, stim_types, dataset_name='dSC_NTSR_CTRL_post_VR_good_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)

LP_NTSR_CNO_pre_dict = get_cluster_frs(LP_NTSR_CNO_keys, stim_types, dataset_name='LP_NTSR_CNO_pre_VR_good_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
LP_NTSR_CNO_post_dict = get_cluster_frs(LP_NTSR_CNO_keys, stim_types, dataset_name='LP_NTSR_CNO_post_VR_good_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
LP_NTSR_CTRL_pre_dict = get_cluster_frs(LP_NTSR_CTRL_keys, stim_types, dataset_name='LP_NTSR_CTRL_pre_VR_good_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
LP_NTSR_CTRL_post_dict = get_cluster_frs(LP_NTSR_CTRL_keys, stim_types, dataset_name='LP_NTSR_CTRL_post_VR_good_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)


sSC_GRP_CNO_keys: 56 , sSC_GRP_CTRL_keys: 82
dSC_GRP_CNO_keys: 119 , dSC_GRP_CTRL_keys: 206
sSC_NTSR_CNO_keys: 30 , sSC_NTSR_CTRL_keys: 33
dSC_NTSR_CNO_keys: 66 , dSC_NTSR_CTRL_keys: 89
LP_NTSR_CNO_keys: 171 , LP_NTSR_CTRL_keys: 139
Data not found locally, extracting from datajoint...
{'experiment_id': '02293', 'mouse_id': '487RR', 'session_id': '20240628', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 316}
Data not found locally, extracting from datajoint...
{'experiment_id': '02293', 'mouse_id': '487RR', 'session_id': '20240628', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 316}
Data not found locally, extracting from datajoint...
{'experiment_id': '02293', 'mouse_id': '487RR', 'session_id': '20240626', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 277}
Data not found locally, extracting from datajoint...
{'experiment_id': '02293', 'mouse_id': '487RR', 

In [6]:
# -------------- STATIC ONLY - VR - ALL - INHIBITION --------------
stim_types = ['EXPA', 'EXPA_60', 'EXPA_25', 'EXPW', 'EXPW_60','DIMM']

ds = datasets.get_dataset('inhibition', as_keys=True)
SC_GRP_CNO_recs_dict = ds[f'GRP_SC_CNO']
SC_GRP_CTRL_recs_dict = ds['GRP_SC_CTRL']
SC_NTSR_CNO_recs_dict = ds[f'NTSR_SC_CNO']
SC_NTSR_CTRL_recs_dict = ds['NTSR_SC_CTRL']
LP_NTSR_CNO_recs_dict = ds[f'NTSR_LP_CNO']
LP_NTSR_CTRL_recs_dict = ds['NTSR_LP_CTRL']

lm_clusters = cluster_table.aggr((cluster_loom_metrics_table), lm_zeta_p="min(lm_zeta_p)")
all_clusters = lm_clusters & 'lm_zeta_p<0.05'

sSC_GRP_CNO_clusters = (cluster_table * cluster_depth_table & SC_GRP_CNO_recs_dict & {'area': 'sSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
sSC_GRP_CTRL_clusters = (cluster_table * cluster_depth_table & SC_GRP_CTRL_recs_dict & {'area': 'sSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_GRP_CNO_clusters = (cluster_table * cluster_depth_table & SC_GRP_CNO_recs_dict & {'area': 'dSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_GRP_CTRL_clusters = (cluster_table * cluster_depth_table & SC_GRP_CTRL_recs_dict & {'area': 'dSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
sSC_NTSR_CNO_clusters = (cluster_table * cluster_depth_table & SC_NTSR_CNO_recs_dict & {'area': 'sSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
sSC_NTSR_CTRL_clusters = (cluster_table * cluster_depth_table & SC_NTSR_CTRL_recs_dict & {'area': 'sSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_NTSR_CNO_clusters = (cluster_table * cluster_depth_table & SC_NTSR_CNO_recs_dict & {'area': 'dSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_NTSR_CTRL_clusters = (cluster_table * cluster_depth_table & SC_NTSR_CTRL_recs_dict & {'area': 'dSC'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
LP_NTSR_CNO_clusters = (cluster_table * cluster_depth_table & LP_NTSR_CNO_recs_dict & {'area': 'LP'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
LP_NTSR_CTRL_clusters = (cluster_table * cluster_depth_table & LP_NTSR_CTRL_recs_dict & {'area': 'LP'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')

sSC_GRP_CNO_keys = (all_clusters & sSC_GRP_CNO_clusters).fetch('KEY')
sSC_GRP_CTRL_keys = (all_clusters & sSC_GRP_CTRL_clusters).fetch('KEY')
dSC_GRP_CNO_keys = (all_clusters & dSC_GRP_CNO_clusters).fetch('KEY')
dSC_GRP_CTRL_keys = (all_clusters & dSC_GRP_CTRL_clusters).fetch('KEY')

sSC_NTSR_CNO_keys = (all_clusters & sSC_NTSR_CNO_clusters).fetch('KEY')
sSC_NTSR_CTRL_keys = (all_clusters & sSC_NTSR_CTRL_clusters).fetch('KEY')
dSC_NTSR_CNO_keys = (all_clusters & dSC_NTSR_CNO_clusters).fetch('KEY')
dSC_NTSR_CTRL_keys = (all_clusters & dSC_NTSR_CTRL_clusters).fetch('KEY')

LP_NTSR_CNO_keys = (all_clusters & LP_NTSR_CNO_clusters).fetch('KEY')
LP_NTSR_CTRL_keys = (all_clusters & LP_NTSR_CTRL_clusters).fetch('KEY')

print("sSC_GRP_CNO_keys:", len(sSC_GRP_CNO_keys), ", sSC_GRP_CTRL_keys:", len(sSC_GRP_CTRL_keys))
print("dSC_GRP_CNO_keys:", len(dSC_GRP_CNO_keys), ", dSC_GRP_CTRL_keys:", len(dSC_GRP_CTRL_keys))
print("sSC_NTSR_CNO_keys:", len(sSC_NTSR_CNO_keys), ", sSC_NTSR_CTRL_keys:", len(sSC_NTSR_CTRL_keys))
print("dSC_NTSR_CNO_keys:", len(dSC_NTSR_CNO_keys), ", dSC_NTSR_CTRL_keys:", len(dSC_NTSR_CTRL_keys))
print("LP_NTSR_CNO_keys:", len(LP_NTSR_CNO_keys), ", LP_NTSR_CTRL_keys:", len(LP_NTSR_CTRL_keys))

sSC_GRP_CNO_pre_dict = get_cluster_frs(sSC_GRP_CNO_keys, stim_types, dataset_name='sSC_GRP_CNO_pre_VR_all_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
sSC_GRP_CNO_post_dict = get_cluster_frs(sSC_GRP_CNO_keys, stim_types, dataset_name='sSC_GRP_CNO_post_VR_all_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
sSC_GRP_CTRL_pre_dict = get_cluster_frs(sSC_GRP_CTRL_keys, stim_types, dataset_name='sSC_GRP_CTRL_pre_VR_all_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
sSC_GRP_CTRL_post_dict = get_cluster_frs(sSC_GRP_CTRL_keys, stim_types, dataset_name='sSC_GRP_CTRL_post_VR_all_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
dSC_GRP_CNO_pre_dict = get_cluster_frs(dSC_GRP_CNO_keys, stim_types, dataset_name='dSC_GRP_CNO_pre_VR_all_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
dSC_GRP_CNO_post_dict = get_cluster_frs(dSC_GRP_CNO_keys, stim_types, dataset_name='dSC_GRP_CNO_post_VR_all_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
dSC_GRP_CTRL_pre_dict = get_cluster_frs(dSC_GRP_CTRL_keys, stim_types, dataset_name='dSC_GRP_CTRL_pre_VR_all_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
dSC_GRP_CTRL_post_dict = get_cluster_frs(dSC_GRP_CTRL_keys, stim_types, dataset_name='dSC_GRP_CTRL_post_VR_all_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)

sSC_NTSR_CNO_pre_dict = get_cluster_frs(sSC_NTSR_CNO_keys, stim_types, dataset_name='sSC_NTSR_CNO_pre_VR_all_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
sSC_NTSR_CNO_post_dict = get_cluster_frs(sSC_NTSR_CNO_keys, stim_types, dataset_name='sSC_NTSR_CNO_post_VR_all_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
sSC_NTSR_CTRL_pre_dict = get_cluster_frs(sSC_NTSR_CTRL_keys, stim_types, dataset_name='sSC_NTSR_CTRL_pre_VR_all_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
sSC_NTSR_CTRL_post_dict = get_cluster_frs(sSC_NTSR_CTRL_keys, stim_types, dataset_name='sSC_NTSR_CTRL_post_VR_all_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
dSC_NTSR_CNO_pre_dict = get_cluster_frs(dSC_NTSR_CNO_keys, stim_types, dataset_name='dSC_NTSR_CNO_pre_VR_all_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
dSC_NTSR_CNO_post_dict = get_cluster_frs(dSC_NTSR_CNO_keys, stim_types, dataset_name='dSC_NTSR_CNO_post_VR_all_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
dSC_NTSR_CTRL_pre_dict = get_cluster_frs(dSC_NTSR_CTRL_keys, stim_types, dataset_name='dSC_NTSR_CTRL_pre_VR_all_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
dSC_NTSR_CTRL_post_dict = get_cluster_frs(dSC_NTSR_CTRL_keys, stim_types, dataset_name='dSC_NTSR_CTRL_post_VR_all_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)

LP_NTSR_CNO_pre_dict = get_cluster_frs(LP_NTSR_CNO_keys, stim_types, dataset_name='LP_NTSR_CNO_pre_VR_all_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
LP_NTSR_CNO_post_dict = get_cluster_frs(LP_NTSR_CNO_keys, stim_types, dataset_name='LP_NTSR_CNO_post_VR_all_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)
LP_NTSR_CTRL_pre_dict = get_cluster_frs(LP_NTSR_CTRL_keys, stim_types, dataset_name='LP_NTSR_CTRL_pre_VR_all_fr25_20260301', means=False, subsession_id='01', fix_sweeping=False)
LP_NTSR_CTRL_post_dict = get_cluster_frs(LP_NTSR_CTRL_keys, stim_types, dataset_name='LP_NTSR_CTRL_post_VR_all_fr25_20260301', means=False, subsession_id='03', fix_sweeping=False)


sSC_GRP_CNO_keys: 322 , sSC_GRP_CTRL_keys: 404
dSC_GRP_CNO_keys: 239 , dSC_GRP_CTRL_keys: 385
sSC_NTSR_CNO_keys: 187 , sSC_NTSR_CTRL_keys: 227
dSC_NTSR_CNO_keys: 123 , dSC_NTSR_CTRL_keys: 178
LP_NTSR_CNO_keys: 208 , LP_NTSR_CTRL_keys: 195
Data not found locally, extracting from datajoint...
{'experiment_id': '02293', 'mouse_id': '487RR', 'session_id': '20240628', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 322}
Data not found locally, extracting from datajoint...
{'experiment_id': '02293', 'mouse_id': '487RR', 'session_id': '20240628', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 322}
Data not found locally, extracting from datajoint...
{'experiment_id': '02293', 'mouse_id': '487RR', 'session_id': '20240626', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 281}
Data not found locally, extracting from datajoint...
{'experiment_id': '02293', 'mouse_id': '48

In [ ]:
# Add extra experiments


In [51]:
# Add optotagging units!
# Detect which ones are optotagged
opto_curation_table & 'response_onset_ms<5'

experiment_id,mouse_id,session_id,preprocessing_type,"session_split full, 01, 02, 03, etc.",sorter_type,cluster_id,label,manual,peak_time_ms,response_onset_ms,response_prob,duration_index,habituation_pct_drop,peak_zscore
01942,74R,20230307,IBL,01,KS3,0,13,Sec trans/adapt,11.5,2.5,0.304,-0.683,0.475,10.26
01942,74R,20230307,IBL,01,KS3,4,13,Sec trans/adapt,11.5,1.5,0.096,-0.652,0.518,8.7
01942,74R,20230307,IBL,01,KS3,31,13,Sec trans/adapt,9.5,2.5,0.752,-0.273,0.368,14.1
01942,74R,20230307,IBL,01,KS3,75,7,Sec sust/noadapt,29.5,4.5,0.568,-0.631,0.23,29.57
01942,74R,20230307,IBL,01,KS3,85,8,Sec sust/noadapt,5.5,4.5,1.0,-0.375,0.151,83.41
01942,74R,20230307,IBL,01,KS3,109,13,Poly-synaptic,13.5,3.5,0.176,-0.475,0.446,7.56
01942,74R,20230307,IBL,01,KS3,122,9,Sec sust/noadapt,23.5,4.5,0.298,-0.622,0.302,24.7
01942,74R,20230307,IBL,01,KS3,124,7,Sec sust/noadapt,25.5,4.5,0.608,-0.568,0.02,11.65
01942,74R,20230307,IBL,01,KS3,132,10,Primary optotagged,6.5,4.5,0.606,0.84,0.299,100.0
01942,74R,20230307,IBL,01,KS3,134,2,Poly-synaptic,3.5,2.92,0.084,0.0,0.103,2.07


In [ ]:
opto_curation_table = ft.ephys_tables.OptoCuration()
stim_types = ['EXPA', 'EXPA_60', 'EXPA_25', 'EXPW', 'EXPW_60', 'DIMM']

# -------------- STATIC ONLY - VR - GOOD --------------
lm_clusters = cluster_table.aggr((cluster_loom_metrics_table), lm_zeta_p="min(lm_zeta_p)")
# sm_clusters = cluster_table.aggr(cluster_sweep_metrics_table, sm_zeta_p="min(sm_zeta_p)")
# all_clusters = lm_clusters * sm_clusters & 'lm_zeta_p<0.05 OR sm_zeta_p<0.05'
all_clusters = lm_clusters & 'lm_zeta_p<0.05'

# sSC_clusters = (opto_curation_table * cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'sSC', 'auto_group': 'good'} & 'label IN (10, 11)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
# dSC_clusters = (opto_curation_table * cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'dSC', 'auto_group': 'good'} & 'label IN (10,11,8,13,7,12,9)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
# LP_clusters = (opto_curation_table * cluster_table * cluster_depth_table & LP_recs_dict & {'area': 'LP', 'auto_group': 'good'} & 'label IN (10,11,8,13,7,12,9)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
sSC_clusters = (opto_curation_table * cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'sSC'} & 'label IN (10, 11)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
dSC_clusters = (opto_curation_table * cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'dSC'} & 'label IN (10,11,8,13,7,12,9)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
LP_clusters = (opto_curation_table * cluster_table * cluster_depth_table & LP_recs_dict & {'area': 'LP'} & 'label IN (10,11,8,13,7,12,9)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')

sSC_keys = (all_clusters & sSC_clusters).fetch('KEY')
dSC_keys = (all_clusters & dSC_clusters).fetch('KEY')
LP_keys = (all_clusters & LP_clusters).fetch('KEY')

print("sSC:", len(sSC_keys), ", dSC:", len(dSC_keys), ", LP:", len(LP_keys))

sSC_dict = get_cluster_frs(sSC_keys, stim_types, dataset_name='sSC_VR_good_opto_fr25_20260729', means=False, subsession_id='01', fix_sweeping=False)
dSC_dict = get_cluster_frs(dSC_keys, stim_types, dataset_name='dSC_VR_good_opto_fr25_20260729', means=False, subsession_id='01', fix_sweeping=False)
# LP_dict = get_cluster_frs(LP_keys, stim_types, dataset_name='LP_VR_good_opto_fr25_20260729', means=False, subsession_id='01', fix_sweeping=False)



sSC: 4 , dSC: 27 , LP: 0
Data not found locally, extracting from datajoint...
{'experiment_id': '02367', 'mouse_id': '157RR', 'session_id': '20240823', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 586}
Data not found locally, extracting from datajoint...
{'experiment_id': '02367', 'mouse_id': '157RR', 'session_id': '20240823', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 564}


In [6]:
opto_curation_table = ft.ephys_tables.OptoCuration()
stim_types = ['EXPA', 'EXPA_60', 'EXPA_25', 'EXPW', 'EXPW_60', 'DIMM']

# -------------- STATIC ONLY - VR - GOOD --------------
lm_clusters = cluster_table.aggr((cluster_loom_metrics_table), lm_zeta_p="min(lm_zeta_p)")
# sm_clusters = cluster_table.aggr(cluster_sweep_metrics_table, sm_zeta_p="min(sm_zeta_p)")
# all_clusters = lm_clusters * sm_clusters & 'lm_zeta_p<0.05 OR sm_zeta_p<0.05'
all_clusters = lm_clusters & 'lm_zeta_p<0.05'

# sSC_clusters = (opto_curation_table * cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'sSC', 'auto_group': 'good'} & 'label IN (10, 11)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
# dSC_clusters = (opto_curation_table * cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'dSC', 'auto_group': 'good'} & 'label IN (10,11,8,13,7,12,9)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
# LP_clusters = (opto_curation_table * cluster_table * cluster_depth_table & LP_recs_dict & {'area': 'LP', 'auto_group': 'good'} & 'label IN (10,11,8,13,7,12,9)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
# sSC_clusters = (opto_curation_table * cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'sSC'} & 'label IN (10, 11)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
# dSC_clusters = (opto_curation_table * cluster_table * cluster_depth_table & SC_recs_dict & {'area': 'dSC'} & 'label IN (10,11,8,13,7,12,9)' & 'fr>0.5').fetch('KEY')#.fetch(format='frame')
LP_clusters = (opto_curation_table * cluster_table * cluster_depth_table & LP_recs_dict & {'area': 'LP'} & 'fr>0.5').fetch('KEY')#.fetch(format='frame')

# sSC_keys = (all_clusters & sSC_clusters).fetch('KEY')
# dSC_keys = (all_clusters & dSC_clusters).fetch('KEY')
LP_keys = (all_clusters & LP_clusters).fetch('KEY')

# print("sSC:", len(sSC_keys), ", dSC:", len(dSC_keys), ", LP:", len(LP_keys))

# sSC_dict = get_cluster_frs(sSC_keys, stim_types, dataset_name='sSC_VR_good_opto_fr25_20260729', means=False, subsession_id='01', fix_sweeping=False)
# dSC_dict = get_cluster_frs(dSC_keys, stim_types, dataset_name='dSC_VR_good_opto_fr25_20260729', means=False, subsession_id='01', fix_sweeping=False)
LP_dict = get_cluster_frs(LP_keys, stim_types, dataset_name='LP_VR_good_opto_fr25_20260810', means=False, subsession_id='01', fix_sweeping=False)



Data not found locally, extracting from datajoint...
{'experiment_id': '02369', 'mouse_id': '426LR', 'session_id': '20240826', 'preprocessing_type': 'IBL', 'session_split': 'full', 'sorter_type': 'KS3', 'cluster_id': 707}


In [ ]:
# print(stim_types)
stim_types = ['EXPA', 'EXPA_60', 'EXPA_25', 'EXPW', 'EXPW_60', 'DIMM']
for key in dSC_keys:
    if np.any([len((trials_table & key & {'stim_type': stim_type}).fetch("KEY")) == 0 for stim_type in stim_types]):
        print("Skipping key: ", key)
        continue


In [41]:
trials_table & {'experiment_id': '01942'}

experiment_id,mouse_id,session_id,subsession_id,preprocessing_type,"session_split full, 01, 02, 03, etc.",sorter_type,trial_id,stim_type,stim_time,clusters,cluster_info,cluster_metrics,length
01942,74R,20230307,01,IBL,01,KS3,0,DIMM_mich,5357530.0,=BLOB=,=BLOB=,=BLOB=,300000.0
01942,74R,20230307,01,IBL,01,KS3,1,EXPA,5893020.0,=BLOB=,=BLOB=,=BLOB=,300000.0
01942,74R,20230307,01,IBL,01,KS3,2,DIMM,6386520.0,=BLOB=,=BLOB=,=BLOB=,300000.0
01942,74R,20230307,01,IBL,01,KS3,3,EXPW_60,6809510.0,=BLOB=,=BLOB=,=BLOB=,300000.0
01942,74R,20230307,01,IBL,01,KS3,4,SWEP_12,7234500.0,=BLOB=,=BLOB=,=BLOB=,300000.0
01942,74R,20230307,01,IBL,01,KS3,5,EXPW_60,7961500.0,=BLOB=,=BLOB=,=BLOB=,300000.0
01942,74R,20230307,01,IBL,01,KS3,6,EXPW_60,8271000.0,=BLOB=,=BLOB=,=BLOB=,300000.0
01942,74R,20230307,01,IBL,01,KS3,7,DIMM,8738990.0,=BLOB=,=BLOB=,=BLOB=,300000.0
01942,74R,20230307,01,IBL,01,KS3,8,EXPW,9189980.0,=BLOB=,=BLOB=,=BLOB=,300000.0
01942,74R,20230307,01,IBL,01,KS3,9,EXPW,9621980.0,=BLOB=,=BLOB=,=BLOB=,300000.0


In [39]:
dSC_keys[-10:]

[{'experiment_id': '02367',
  'mouse_id': '157RR',
  'session_id': '20240823',
  'preprocessing_type': 'IBL',
  'session_split': 'full',
  'sorter_type': 'KS3',
  'cluster_id': 553},
 {'experiment_id': '02367',
  'mouse_id': '157RR',
  'session_id': '20240823',
  'preprocessing_type': 'IBL',
  'session_split': 'full',
  'sorter_type': 'KS3',
  'cluster_id': 554},
 {'experiment_id': '02367',
  'mouse_id': '157RR',
  'session_id': '20240823',
  'preprocessing_type': 'IBL',
  'session_split': 'full',
  'sorter_type': 'KS3',
  'cluster_id': 558},
 {'experiment_id': '02367',
  'mouse_id': '157RR',
  'session_id': '20240823',
  'preprocessing_type': 'IBL',
  'session_split': 'full',
  'sorter_type': 'KS3',
  'cluster_id': 560},
 {'experiment_id': '02367',
  'mouse_id': '157RR',
  'session_id': '20240823',
  'preprocessing_type': 'IBL',
  'session_split': 'full',
  'sorter_type': 'KS3',
  'cluster_id': 562},
 {'experiment_id': '02367',
  'mouse_id': '157RR',
  'session_id': '20240823',
  'pre

In [36]:
lm_clusters * opto_curation_table * cluster_table * cluster_depth_table & {'experiment_id': '01942'} & {'area': 'dSC', 'auto_group': 'good'} & 'label IN (10,11,8,13,7,12,9)' & 'fr>0.5'

experiment_id,mouse_id,session_id,preprocessing_type,"session_split full, 01, 02, 03, etc.",sorter_type,cluster_id,shank,lm_zeta_p calculated attribute,label,manual,peak_time_ms,response_onset_ms,response_prob,duration_index,habituation_pct_drop,peak_zscore,amplitude,contam_pct,ks_label,amp,ch,depth,fr,group,n_spikes,sh,firing_rate,presence_ratio,isi_viol,amplitude_cutoff,isi_ratio,tracked,auto_group,cluster_depth,area
01942,74R,20230307,IBL,01,KS3,2,2,0.242651,13,Sec trans/adapt,10.5,9.5,0.67,-0.659,0.406,12.92,1865.1,100.0,good,76.0468,56,1500.0,20.0912,-1.0,79826,0.0,20.0373,0.99,0.0169316,5.04338e-05,0.00125628,1,good,1140,dSC
01942,74R,20230307,IBL,01,KS3,5,2,0.0809457,7,Sec trans/noadapt,10.5,10.5,0.356,-1.322,0.273,6.47,1633.3,100.0,good,94.9166,57,1500.0,12.0302,-1.0,47798,0.0,11.8694,0.99,0.0167541,0.00176918,0.000869528,1,good,1140,dSC
01942,74R,20230307,IBL,01,KS3,6,2,0.0903321,13,Sec trans/adapt,10.5,9.5,0.288,-0.75,0.534,10.25,2323.2,100.0,good,97.8293,59,1515.0,4.9132,-1.0,19521,0.0,4.82935,0.99,0.00404542,0.000936465,5.21241e-05,1,good,1125,dSC
01942,74R,20230307,IBL,01,KS3,11,2,0.094343,13,Sec trans/adapt,11.5,8.5,0.442,-0.636,0.501,15.96,1128.9,100.0,mua,66.9099,61,1530.0,17.2801,-1.0,68657,0.0,17.182,0.99,0.0818174,0.000199159,0.00367728,1,good,1110,dSC
01942,74R,20230307,IBL,01,KS3,12,2,0.116538,7,Sec sust/adapt,10.5,5.5,0.912,-0.485,0.285,5.81,2351.0,100.0,good,88.6917,60,1530.0,46.2635,-1.0,183813,0.0,45.7987,0.99,0.00337522,8.78418e-05,0.000648569,1,good,1110,dSC
01942,74R,20230307,IBL,01,KS3,16,2,0.0470049,9,Poly-synaptic,15.5,15.5,0.046,-1.75,0.083,2.5,1476.5,100.0,good,91.9126,64,1560.0,1.10692,-1.0,4398,0.0,1.10029,0.99,0.0,0.0125905,0.0,1,good,1080,dSC
01942,74R,20230307,IBL,01,KS3,20,2,0.154633,13,Sec trans/adapt,11.5,8.5,0.158,-0.889,0.488,9.98,1253.6,100.0,mua,73.9813,61,1530.0,4.79491,-1.0,19051,0.0,4.77976,0.99,0.210664,0.00100976,0.00273857,1,good,1110,dSC
01942,74R,20230307,IBL,01,KS3,30,2,0.0179027,7,Poly-synaptic,13.5,6.5,0.77,-0.754,0.0,14.26,3207.2,100.0,good,123.29,70,1605.0,13.1016,-1.0,52055,0.0,12.6905,0.99,0.00117174,0.000210962,5.95073e-05,1,good,1035,dSC
01942,74R,20230307,IBL,01,KS3,31,2,0.354518,13,Sec trans/adapt,9.5,2.5,0.752,-0.273,0.368,14.1,1455.8,100.0,good,85.1549,68,1590.0,20.4323,-1.0,81181,0.0,20.2249,0.99,0.0133792,3.22089e-06,0.000759226,1,good,1050,dSC
01942,74R,20230307,IBL,01,KS3,33,2,0.0557289,7,Poly-synaptic,10.5,9.5,0.804,-0.847,0.089,7.95,2157.9,100.0,good,97.7684,77,1650.0,15.9507,-1.0,63375,0.0,15.9423,0.99,0.00445474,0.000108321,0.000173688,1,good,990,dSC


In [21]:
subsess_ids

array(['01_OPTS03', '01_OPTS04', '03_OPTS02', '03_OPTS03', '03_OPTS04'],
      dtype=object)

In [14]:
opto_table & key

experiment_id,mouse_id,session_id,subsession_id,preprocessing_type,"session_split full, 01, 02, 03, etc.",sorter_type,idxs,opto_metrics
02030,255L,20230720,01_OPTS03,IBL,full,KS3,=BLOB=,=BLOB=
02030,255L,20230720,01_OPTS04,IBL,full,KS3,=BLOB=,=BLOB=
02030,255L,20230720,03_OPTS02,IBL,full,KS3,=BLOB=,=BLOB=
02030,255L,20230720,03_OPTS03,IBL,full,KS3,=BLOB=,=BLOB=
02030,255L,20230720,03_OPTS04,IBL,full,KS3,=BLOB=,=BLOB=
